# Session 9 — PBDW and observable background spaces

Download the notebook with the toolbar. Run the supplied baseline from a fresh kernel before changing settings.
Use [the course Python environment](https://feelpp.github.io/course-rom/course-rom/setup.html). Each practical starts independently of your earlier notebooks.
Read [the accompanying notes](https://feelpp.github.io/course-rom/rom/assimilation/variational.html) for assumptions and derivations.
The timed tasks below occupy 60 minutes, including the closing comparison; optional extensions are outside that budget.
Website plots come from executing these same cells. Synthetic truth is used to evaluate methods, never as an undeclared estimator input.
## Background and Riesz map (15 minutes)

The background is a rank-three POD space. The held-out state includes an added smooth discrepancy.
The Euclidean measurement norm and mass-like state metric $G=hI$ are distinct.
**Task 1.** Check the Riesz identity using a random state before solving the saddle system.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
n=100
h=1/n
x=(np.arange(n)+.5)*h
def field(c,w=.18): return np.exp(-((x-c)/w)**2)
S=np.column_stack([field(c) for c in np.linspace(.25,.75,15)])
def dictionary(centers,width=.07):
    H=np.exp(-((x[None,:]-np.asarray(centers)[:,None])/width)**2)
    return H/H.sum(axis=1,keepdims=True)  # Quadrature-normalized averages.
Hdict=dictionary(np.linspace(.05,.95,20))

U,s,_=np.linalg.svd(np.sqrt(h)*S,full_matrices=False)
r=3
Z=U[:,:r]/np.sqrt(h)
H=dictionary(np.linspace(.08,.92,8))
def beta_for(H,Z):
    if np.linalg.matrix_rank(H@Z)<Z.shape[1]: return 0.
    Wbar=np.linalg.qr(H.T/np.sqrt(h),mode='reduced')[0]
    Vbar=np.linalg.qr(np.sqrt(h)*Z,mode='reduced')[0]
    return float(np.linalg.svd(Wbar.T@Vbar,compute_uv=False)[-1])
def pbdw(H,Z,y,xi=0.):
    Q=H.T/h
    A=H@Q; B=H@Z
    saddle=np.block([[A+xi*np.eye(len(H)),B],[B.T,np.zeros((Z.shape[1],Z.shape[1]))]])
    solution=np.linalg.solve(saddle,np.r_[y,np.zeros(Z.shape[1])])
    d,a=solution[:len(H)],solution[len(H):]
    return Z@a+Q@d,Q@d,d
truth=field(.43,.16)+.12*np.sin(3*np.pi*x)


## Exact-data assimilation (15 minutes)

**Task 2.** Verify both blocks of the saddle equations. Interpret correction orthogonality and exact observation fitting.
The best background projection below is an oracle diagnostic using truth, not an operational competing estimator.


In [ ]:
y=H@truth
estimate,update,d=pbdw(H,Z,y)
projection=Z@(h*Z.T@truth)
beta=beta_for(H,Z)
print('PBDW beta:',beta)
print('Riesz identity defect:',np.linalg.norm(H-(H.T/h).T*h))
print('Observation residual:',np.linalg.norm(H@estimate-y))
print('Update/background orthogonality:',np.linalg.norm(h*Z.T@update))
print('PBDW/oracle projection errors:',np.sqrt(h)*np.linalg.norm(estimate-truth),np.sqrt(h)*np.linalg.norm(projection-truth))


## Interpret observability (20 minutes)

**Task 3.** Rescale background columns by different nonzero constants and recompute beta. Repeat with clustered sensor centers.
Do not identify beta with the smallest singular value of the raw observation/background matrix.


In [ ]:
fig,ax=plt.subplots(figsize=(7,3.5))
ax.plot(x,truth,label='Held-out state'); ax.plot(x,projection,'--',label='Oracle background projection')
ax.plot(x,estimate,':',label='PBDW'); ax.set(xlabel='x',ylabel='State')
ax.legend(fontsize=8); fig.tight_layout(); plt.show()
print('Rescaled-background beta:',beta_for(H,Z@np.diag([.1,2,10])))
print('Clustered-sensor beta:',beta_for(dictionary(np.linspace(.05,.35,8)),Z))


## Checkpoint (10 minutes)

Submit the Riesz, orthogonality and data checks, plus the two beta values.
Explain how an invisible discrepancy limits reconstruction despite exact data fitting.
Optional: construct an exact state in the background and confirm that PBDW recovers it with zero update.
